# Assignment 6

To measure the psychological state of Americans, a psychology researcher (in 1929) devised
three questions put to 3443 Americans from different walks of life. He asked them whether they
were happy. They could answer only **yes** or **no** to each question. By today’s standards the study
is quite simple, and he did no formal statistical analysis of his data. The frequency data (number
of **yes** and **no** responses within each employment category) from this study are in the file
*Americandata.csv*.

---

You need to use the data and answer the question, that is, **Is the type of profession related to the happiness of American people?**
Report the **χ2** statistic and the *statistical significance of test*, in addition to the contingency table.
Also, explain the results and what they mean. What does the value of **χ2** statistic mean? Is it
good or bad? Is the test statistically significant?

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# Load the contingency table (rows = Yes/No happy, columns = profession)
df = pd.read_csv("./Americandata.csv", index_col="Happy")
print("Observed contingency table:")
print(df)
print(f"\nTotal respondents: {df.values.sum()}")

Observed contingency table:
       Students  Laborers  Preachers  Physicians  Housewives  Teachers  \
Happy                                                                    
Yes         390       378         35         159          78       108   
No         1610       122        265          51         122        38   

       Lawyers  Musicians  
Happy                      
Yes         11         31  
No          64         19  

Total respondents: 3481


In [2]:
# Contingency table with row/column totals
table = df.copy()
table["Total"] = table.sum(axis=1)
table.loc["Total"] = table.sum()
table

,Students,Laborers,Preachers,Physicians,Housewives,Teachers,Lawyers,Musicians,Total
Happy,,,,,,,,,
Yes,390,378,35,159,78,108,11,31,1190
No,1610,122,265,51,122,38,64,19,2291
Total,2000,500,300,210,200,146,75,50,3481


In [3]:
# Percentage of each profession answering Yes / No (column percentages)
col_pct = (df / df.sum() * 100).round(1)
col_pct

,Students,Laborers,Preachers,Physicians,Housewives,Teachers,Lawyers,Musicians
Happy,,,,,,,,
Yes,19.5,75.6,11.7,75.7,39.0,74.0,14.7,62.0
No,80.5,24.4,88.3,24.3,61.0,26.0,85.3,38.0


In [4]:
# Perform Chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(df)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p:.3e}")
print(f"Degrees of freedom: {dof}")
print(f"Critical value (alpha = 0.05): {stats.chi2.ppf(0.95, dof):.2f}")

Chi-square statistic: 936.14
p-value: 7.523e-198
Degrees of freedom: 7
Critical value (alpha = 0.05): 14.07


In [5]:
# Expected frequencies if profession and happiness were independent
expected_df = pd.DataFrame(expected, index=df.index, columns=df.columns).round(1)
expected_df

,Students,Laborers,Preachers,Physicians,Housewives,Teachers,Lawyers,Musicians
Happy,,,,,,,,
Yes,683.7,170.9,102.6,71.8,68.4,49.9,25.6,17.1
No,1316.3,329.1,197.4,138.2,131.6,96.1,49.4,32.9


In [6]:
# Standardized residuals: (observed - expected) / sqrt(expected)
# |value| > 2 marks a cell that differs notably from what independence predicts
residuals = ((df - expected) / np.sqrt(expected)).round(2)
residuals

,Students,Laborers,Preachers,Physicians,Housewives,Teachers,Lawyers,Musicians
Happy,,,,,,,,
Yes,-11.23,15.84,-6.67,10.29,1.16,8.22,-2.89,3.36
No,8.10,-11.42,4.81,-7.42,-0.84,-5.93,2.08,-2.42


In [7]:
# Effect size: Cramer's V
n = df.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(df.shape) - 1)))
print(f"Cramer's V: {cramers_v:.3f}")

Cramer's V: 0.519


Put your analysis in the empty `markdown` cell below.

## Analysis

**Question:** Is the type of profession related to the happiness of American people?

**Hypotheses**
- H₀: Profession and happiness are independent (not related).
- H₁: Profession and happiness are related.

**Contingency table** (observed counts; % answering *Yes* in the last row)

| Happy | Students | Laborers | Preachers | Physicians | Housewives | Teachers | Lawyers | Musicians | Total |
|---|---|---|---|---|---|---|---|---|---|
| Yes | 390 | 378 | 35 | 159 | 78 | 108 | 11 | 31 | 1190 |
| No | 1610 | 122 | 265 | 51 | 122 | 38 | 64 | 19 | 2291 |
| Total | 2000 | 500 | 300 | 210 | 200 | 146 | 75 | 50 | 3481 |
| % Yes | 19.5% | 75.6% | 11.7% | 75.7% | 39.0% | 74.0% | 14.7% | 62.0% | 34.2% |

*(Note: the counts in the file add up to 3,481 people, not the 3,443 given in the description.)*

**Results**
- χ² = **936.14**
- Degrees of freedom = (2 − 1)(8 − 1) = **7**
- p-value ≈ **7.5 × 10⁻¹⁹⁸** (effectively 0)
- Critical value at α = 0.05 with 7 df = 14.07
- Cramér's V = **0.52**

**What the χ² value means.** The χ² statistic measures how far the observed counts are from the counts we would expect if profession had no relationship to happiness. It adds up (observed − expected)² / expected across every cell. A value near 0 means the data look like independence. A large value means the observed pattern is very different from independence. Our χ² of 936 is far larger than the critical value of 14.07, so the difference is far too large to be due to chance.

**Is it good or bad?** χ² is not "good" or "bad" in itself. It only measures how strong the evidence against independence is. Here the large value is "good" for answering the research question, because it gives strong evidence that profession and happiness *are* related. It does not say which professions are happier. For that we look at the table and the residuals.

**Is the test statistically significant?** Yes. p ≈ 7.5 × 10⁻¹⁹⁸ is far below α = 0.05, so we **reject H₀**. Cramér's V = 0.52 indicates a **strong** association, so the result is also meaningful in practice, not just because the sample is large.

**Interpretation (which professions drive the result).** Overall, 34.2% of respondents said they were happy. Based on the standardized residuals:
- **Happier than expected:** Laborers (75.6% yes, residual +15.8), Physicians (75.7%, +10.3), Teachers (74.0%, +8.2), and Musicians (62.0%, +3.4).
- **Less happy than expected:** Students (19.5% yes, −11.2), Preachers (11.7%, −6.7), and Lawyers (14.7%, −2.9).
- **Housewives** (39.0%) are close to the overall rate (residual +1.2), so they are not unusual.

Laborers contribute the most to χ² (about 381 of the 936), followed by Students (192) and Physicians (161). Housewives contribute almost nothing (2.1).

**Conclusion:** Yes, the type of profession is significantly related to happiness among the Americans surveyed (χ²(7, N = 3481) = 936.14, p < .001, Cramér's V = 0.52). Laborers, physicians and teachers were the most likely to report being happy, while students, preachers and lawyers were the least likely. Because this is an observational survey, the test shows an association, not that a profession causes happiness or unhappiness.